### Motivation behind using transformers
As compared to RNNs, which compute sequentially, with every step h_(n+1) depending on step h_(n) (i.e ht​=f(ht−1​,xt​)), in Transformers, operations can run in parallel as every token attends to every other token in parallel

* Attention matrix can be computed in on batched matrix multiplication without any step inter-dependency

#### Other benefits of using Transformers as compared to RNNs

* Better long range dependency handling -> RNNs struggle with remembering information far earlier in the sequence whereas in Transformers, this issue is mitigated with the help of self-attention which created direct connection between any two tokens

* Attention mechanism learns the context better -> RNNs attempt to compress all the information whereas in Transformers, the words of importance are dynamically chosen

#### Sequential depth vs Total Work Complexity

* Total Work Complexity: Standard Big-O that counts the total number of operations/ all work done
* Sequential Depth: Asks a different question on "How many seqential rounds are required if I have infinite processory"


e.g In the case of a sequential sum of ((((x1​+x2​)+x3​)+x4​)+…), the total work complexity is O(N) as the depth is O(N). While the sequential depth with tree reduction is O(log n), as the ((((x1​+x2​)+x3​)+x4​)+…) operation can be parallelised into (x1+x2), (x3+x4), (x5+x6)

In Transformers, the key operation is the Q*Kt, with Q = N x d matrix and K = N x d matrix, creating a N x N matrix, so work complexity is O(N**2)

However, if enough processors exists, matrix multiplication step can be executed as one parallel step leading to a dependency depth (with respect to sequence length) of O(1). In reality closer to O(log n) as it's not pracially one-step due to hardware performing operations like memory movement and alogrithms using tiling, reduction trees and parallel scans.

#### Demonstrating the serial depth gap between RNN-style recurrence and attention-style parallel execution

In [1]:
import math 
import time

In [ ]:
def rnn_style(xs, decay = 0.9):
    """Sequential recurrence: h_t depends on h_{t-1}. Cannot parallelize."""
    ht = 0
    for xt in xs:
        ht = decay * ht + xt
    return ht

In [ ]:
def attention_style(xs):
    """Order-independent reduction: every element is independent."""
    return sum(xs) / len(xs)

In [4]:
def serial_scan(xs):
    """Prefix sum computed serially. Depth O(N)."""
    out = []
    acc = 0.0
    for x in xs:
        acc += x
        out.append(acc)
    return out


def parallel_scan(xs):
    """Hillis-Steele parallel prefix sum. Depth O(log N).

    In pure Python each step is still serial, but the data-dependency
    graph has depth log2(N). On real hardware with N-wide SIMD this
    gets you a log-depth scan; on a CPU it's the same wall-clock but
    the graph shape is what matters for GPU kernels.
    """
    out = list(xs)
    step = 1
    n = len(out)
    while step < n:
        new = list(out)
        for i in range(step, n):
            new[i] = out[i] + out[i - step]
        out = new
        step *= 2
    return out



In [7]:
def benchmark(n, reps=3):
    xs = [0.001 * (i % 17) for i in range(n)]

    best_rnn = math.inf
    for _ in range(reps):
        t0 = time.perf_counter()
        _ = rnn_style(xs)
        best_rnn = min(best_rnn, time.perf_counter() - t0)

    best_attn = math.inf
    for _ in range(reps):
        t0 = time.perf_counter()
        _ = attention_style(xs)
        best_attn = min(best_attn, time.perf_counter() - t0)

    return best_rnn, best_attn


def depth(n):
    """Serial-depth count for RNN vs attention-style reductions."""
    rnn_depth = n
    attn_depth = max(1, math.ceil(math.log2(n)))
    return rnn_depth, attn_depth


def main():
    print("=== serial-depth comparison ===")
    print(f"{'N':>8}  {'rnn depth':>12}  {'attn depth':>12}  {'speedup (ops)':>16}")
    for n in [64, 512, 4096, 32768, 262144]:
        rd, ad = depth(n)
        print(f"{n:>8}  {rd:>12}  {ad:>12}  {rd / ad:>15.0f}x")

    print()
    print("=== wall-clock on this machine (pure Python) ===")
    print(f"{'N':>8}  {'rnn (ms)':>10}  {'attn (ms)':>10}  {'ratio':>8}")
    for n in [1_000, 10_000, 100_000, 1_000_000]:
        rnn_t, attn_t = benchmark(n)
        ratio = rnn_t / attn_t if attn_t > 0 else float("inf")
        print(f"{n:>8}  {rnn_t * 1000:>10.2f}  {attn_t * 1000:>10.2f}  {ratio:>7.1f}x")

    print()
    print("=== prefix-sum equivalence check ===")
    xs = [float(i) for i in range(16)]
    ser = serial_scan(xs)
    par = parallel_scan(xs)
    mismatches = sum(1 for a, b in zip(ser, par) if abs(a - b) > 1e-9)
    print(f"length: {len(xs)}, mismatches between serial and parallel scan: {mismatches}")
    print(f"last value (serial):   {ser[-1]}")
    print(f"last value (parallel): {par[-1]}")

    print()
    print("takeaway: attention wins on every dimension but memory.")
    print("memory cost is O(N^2) for full attention; Lesson 12 covers the fixes.")


In [8]:
main()

=== serial-depth comparison ===
       N     rnn depth    attn depth     speedup (ops)
      64            64             6               11x
     512           512             9               57x
    4096          4096            12              341x
   32768         32768            15             2185x
  262144        262144            18            14564x

=== wall-clock on this machine (pure Python) ===
       N    rnn (ms)   attn (ms)     ratio
    1000        0.04        0.00     13.6x
   10000        0.14        0.02      6.1x
  100000        1.40        0.21      6.6x
 1000000       11.31        2.17      5.2x

=== prefix-sum equivalence check ===
length: 16, mismatches between serial and parallel scan: 0
last value (serial):   120.0
last value (parallel): 120.0

takeaway: attention wins on every dimension but memory.
memory cost is O(N^2) for full attention; Lesson 12 covers the fixes.
